# **Visualizing World Model results** 

This notebook implements the analysis of the results obtained from our world model.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from jepacartpole.models import VAE, JEPAPredictor
from jepacartpole.config import ConfigVAE, ConfigJEPA

# loading VAE
vae_config = ConfigVAE()
vae = VAE(vae_config).to(DEVICE)
vae.load_state_dict(torch.load('../checkpoints/vae.pth', weights_only=False, map_location=DEVICE))

# loading jepa
jepa_config = ConfigJEPA()
jepa = JEPAPredictor(jepa_config).to(DEVICE)
jepa.load_state_dict(torch.load('../checkpoints/jepa.pth', weights_only=False, map_location=DEVICE))

## 1. Visualizing Trajectory on Latent Space with PCA

In [ ]:
# CELL 1: Latent Space Trajectory Visualization
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from sklearn.decomposition import PCA
import torch
from IPython.display import HTML

# Set to evaluation mode
jepa.eval()
vae.eval() 

# Get a test episode (use first episode from dataset)
test_episode_idx = 0
with h5py.File('../data/jepa/jepa.h5', 'r') as h5f:
    episode_length = h5f[f'episode_{test_episode_idx}_length'][()]
    z_real_full = h5f['z'][test_episode_idx, :episode_length]
    actions = h5f['actions'][test_episode_idx, :episode_length]
    
    # Convert to tensors
    z_real_full = torch.from_numpy(z_real_full).float()
    actions = torch.from_numpy(actions).float()

# Generate predicted trajectory
z_predicted = [z_real_full[0]]  # Start with real first latent

with torch.no_grad():
    for t in range(len(actions) - 1):
        z_t = z_predicted[-1].to(device)
        action_t = actions[t].to(device)
        
        # Predict next latent
        z_next_pred = jepa(z_t.unsqueeze(0), action_t.unsqueeze(0))
        z_predicted.append(z_next_pred.squeeze().cpu())

z_predicted = torch.stack(z_predicted)

# Project to 2D using PCA
pca = PCA(n_components=2)
z_real_2d = pca.fit_transform(z_real_full.numpy())
z_pred_2d = pca.transform(z_predicted.numpy())

# Create animation
fig, ax = plt.subplots(figsize=(10, 8))

# Plot the full trajectories as faint lines
ax.plot(z_real_2d[:, 0], z_real_2d[:, 1], 'b-', alpha=0.3, label='Real trajectory')
ax.plot(z_pred_2d[:, 0], z_pred_2d[:, 1], 'r-', alpha=0.3, label='Predicted trajectory')

# Scatter points for current positions
real_point = ax.scatter([], [], c='blue', s=100, edgecolors='white', linewidth=2, label='Real position')
pred_point = ax.scatter([], [], c='red', s=100, edgecolors='white', linewidth=2, label='Predicted position')

# Add start marker
ax.scatter(z_real_2d[0, 0], z_real_2d[0, 1], c='green', s=200, marker='*', label='Start')

ax.set_xlabel('PC1', fontsize=12)
ax.set_ylabel('PC2', fontsize=12)
ax.set_title('Latent Space Trajectory: Real vs Predicted', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

# Animation function
def animate(frame):
    real_point.set_offsets([z_real_2d[frame, 0], z_real_2d[frame, 1]])
    pred_point.set_offsets([z_pred_2d[frame, 0], z_pred_2d[frame, 1]])
    ax.set_title(f'Step {frame}/{len(z_real_2d)-1}', fontsize=14)
    return real_point, pred_point

# Create and display animation
anim = FuncAnimation(fig, animate, frames=len(z_real_2d), interval=50, blit=True, repeat=True)
HTML(anim.to_html5_video())

# Optional: Save as GIF (uncomment if you have pillow installed)
# anim.save('latent_trajectory.gif', writer='pillow', fps=20)